--- 
# **[실습]**

### 실습 목표

Contextual Retrieval을 실제 문서에 적용하여 검색 성능을 개선합니다.

### 난이도별 가이드

**기본 난이도:**
- 제공된 샘플 문서에 Contextual Retrieval 적용
- 일반 검색 vs Contextual 검색 성능 비교
- 3개 이상의 쿼리로 테스트

**중급 난이도:**
- 자체 문서(예: 기술 문서, 위키피디아) 활용
- Hybrid 검색 (Embedding + BM25) 구현
- 가중치 조합 실험 (0.3:0.7, 0.5:0.5, 0.7:0.3)

**고급 난이도:**
- 다국어 문서에 Contextual Retrieval 적용
- Reranker와 결합하여 최적 파이프라인 구성
- 검색 성능 지표 (HitRate, MRR) 측정 및 비교

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from pprint import pprint
from typing import List, Tuple

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(1) 문서 준비 및 청킹`

자신의 문서를 로드하고 청크로 분할합니다.

In [ ]:
import os
import re
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# 1. 파일 경로 및 로더 지정
file_path = "./data/industry_report/"

loader = DirectoryLoader(
    file_path,
    glob="*.pdf",           # 하위 폴더 미포함 / 포함 : **/*.pdf
    loader_cls=PyMuPDFLoader  
)

# 2. 문서 로드
docs = loader.load()

# ==========================================
# 3. 파일명 기반 메타데이터 동적 추출 및 할당 (추가된 부분)
# ==========================================
# 정규표현식 패턴: [기업명]보고서종류(YYYY.MM.DD).pdf
# 설명: \[ (대괄호 열기) / (.*?) (아무글자나 매칭) / \] (대괄호 닫기)
pattern = r"\[(.*?)\](.*?)\((.*?)\)\.pdf"

for doc in docs:
    # 전체 경로에서 파일명만 추출 (예: ./data/industry_report/[삼성전자]분기보고서(2026.05.15).pdf -> [삼성전자]분기보고서(2026.05.15).pdf)
    source_path = doc.metadata.get('source', '')
    file_name = os.path.basename(source_path)
    
    # 정규식 패턴과 파일명 매칭
    match = re.match(pattern, file_name)
    
    if match:
        company_name = match.group(1)  # 삼성전자
        report_type = match.group(2)   # 분기보고서
        report_date = match.group(3)   # 2026.05.15
        
        # Document 객체의 metadata 딕셔너리에 추출한 값 주입
        doc.metadata['company'] = company_name
        doc.metadata['report_type'] = report_type
        doc.metadata['date'] = report_date
        
        # (선택 팁) 날짜에서 '연도'만 분리해서 메타데이터로 빼두면, 
        # 나중에 "2026년도 리포트만 찾아줘" 할 때 DB 필터링이 훨씬 수월합니다.
        if "." in report_date:
            doc.metadata['year'] = report_date.split('.')[0] 
            
    else:
        # 패턴에 맞지 않는 예외 파일이 있을 경우의 처리
        doc.metadata['company'] = "Unknown"
        doc.metadata['report_type'] = "Unknown"
        print(f"경고: 파일명 패턴 불일치 (메타데이터 추출 실패) -> {file_name}")

# ==========================================
# 4. 로드 및 메타데이터 주입 결과 확인
# ==========================================
print("="*80)
print(f"로드된 총 페이지(Document) 수: {len(docs)}")
print("="*80)

if docs:
    print(f"첫 번째 페이지 출처 (원본): {docs[0].metadata['source']}")
    print("-" * 40)
    print(f"✅ 주입된 메타데이터 정보:\n{docs[0].metadata}")
    print("-" * 40)
    print(f"첫 번째 페이지 내용 일부:\n{docs[0].page_content[:200]}...")

로드된 총 페이지(Document) 수: 1562
첫 번째 페이지 출처 (원본): data\industry_report\[삼성전자]분기보고서(2026.05.15).pdf
----------------------------------------
✅ 주입된 메타데이터 정보:
{'producer': 'iText® 5.4.0 ©2000-2012 1T3XT BVBA (AGPL-version)', 'creator': '', 'creationdate': '2026-05-15T16:14:23+09:00', 'source': 'data\\industry_report\\[삼성전자]분기보고서(2026.05.15).pdf', 'file_path': 'data\\industry_report\\[삼성전자]분기보고서(2026.05.15).pdf', 'total_pages': 323, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-15T16:14:23+09:00', 'trapped': '', 'modDate': "D:20260515161423+09'00'", 'creationDate': "D:20260515161423+09'00'", 'page': 0, 'company': '삼성전자', 'report_type': '분기보고서', 'date': '2026.05.15', 'year': '2026'}
----------------------------------------
첫 번째 페이지 내용 일부:
목                 차
분 기 보 고 서............................................................................................................................................1
【 대표이사 등의 확인 】..................


In [5]:
# ==========================================
# [추가] 목차(TOC)와 본문 동적 분리 로직
# ==========================================
toc_docs = []
body_docs = []

# 정규식 패턴: '점선이나 넓은 공백 뒤에 숫자가 오는 패턴'을 찾습니다. 
# 증권사 리포트 목차에서 흔히 보이는 "산업 동향 ........ 5" 또는 "수주 분석      12" 패턴을 잡기 위함입니다.
toc_pattern = re.compile(r'(?:\.{3,}|\s{4,})\d+') 

for doc in docs:
    content = doc.page_content
    
    # 1. 명시적 키워드 확인 (페이지 상단 200자 이내에 키워드가 있는지)
    header_text = content[:200].lower()
    has_toc_keyword = "목차" in header_text or "contents" in header_text or "index" in header_text
    
    # 2. 정규식 패턴 확인 (목차 패턴이 해당 페이지에 3번 이상 반복되는가?)
    # 일반 본문에도 점과 숫자가 우연히 섞일 수 있으므로, 3번 이상 반복될 때만 목차로 간주합니다.
    pattern_matches = len(toc_pattern.findall(content))
    
    # 판단 로직: 목차 키워드가 있거나, 목차 패턴이 확연히 나타나면 TOC로 분류
    if has_toc_keyword or pattern_matches >= 3:
        doc.metadata['is_toc'] = True
        toc_docs.append(doc)
    else:
        doc.metadata['is_toc'] = False
        body_docs.append(doc)

# 향후 Contextual Retrieval을 위해 목차 텍스트들만 따로 묶어서 딕셔너리로 보관 (선택 사항)
# key: 파일명, value: 해당 파일의 전체 목차 텍스트
toc_text_map = {}
for doc in toc_docs:
    filename = doc.metadata.get('source', 'unknown')
    if filename not in toc_text_map:
        toc_text_map[filename] = ""
    toc_text_map[filename] += doc.page_content + "\n"

# ==========================================
# 4. 분리 결과 확인 및 청킹
# ==========================================
print(f"✅ 분류 완료: 총 {len(docs)}페이지 중 목차 {len(toc_docs)}페이지, 본문 {len(body_docs)}페이지")

# 이제 '목차가 제거된' 순수 본문(body_docs)만 청커에 집어넣습니다!
# chunks = text_splitter.split_documents(body_docs)

✅ 분류 완료: 총 1562페이지 중 목차 132페이지, 본문 1430페이지


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 작은 청크로 분할
    chunk_overlap=50,    # 50자 중복
    separators=["\n\n", "\n", ". ",],
)

# 문서 분할
chunks = text_splitter.split_documents(body_docs)

print(f"생성된 청크 수: {len(chunks)}")
print("="*80)

for i, chunk in enumerate(chunks[:5]):
    print(f"\n[청크 {i+1}] ({len(chunk.page_content)}자)")
    print("-"*40)
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)

생성된 청크 수: 4183

[청크 1] (416자)
----------------------------------------
분 기 보 고 서
 
 
 
 
                                    (제 40 기) 
 
사업연도
2026년 01월 01일
부터
2026년 03월 31일
까지
금융위원회
한국거래소 귀중
2026년    05월    15일
제출대상법인 유형 :
주권상장법인
면제사유발생 :
해당사항 없음
회      사      명 :
(주)엘지씨...

[청크 2] (46자)
----------------------------------------
【 대표이사 등의 확인 】
 
전자공시시스템 dart.fss.or.kr
Page 2

[청크 3] (455자)
----------------------------------------
II. 사업의 내용
 
1. 사업의 개요
 
당사와 연결종속회사가 영위하고 있는 주된 사업은 IT서비스로 클라우드&AI, 스마트 엔지
니어링, Digital Business Service(SI/SM)로 구분할 수 있습니다.
 
(1) 클라우드&AI
 
기업의 클라우드/AI 도입과 활용은 비즈니스 전 밸류체인으로 빠르게 확산되고 있습니다. 특
히 AI가 사람...

[청크 4] (454자)
----------------------------------------
라우드&AI 사업 역량을 고도화하며 AX(AI Transformation) 리더십을 강화해 나가고 있습
니다. 컨설팅부터 클라우드 전환/구축, MSP(Managed Service Provider), 애플리케이션 현
대화(Application Modernization, AM)까지 통합적인 서비스를 제공하며, 우수한 기술 역량
을 바탕으로 고객의 클라우드 이용...

[청크 5] (479자)
----------------------------------------
, 시티 영역에서의 일상생활 혁신을 위한 사업을 지속적으로 추진하고 확대해 왔습니다.

`(2) 컨텍스트 생성`

각 청크에 대해 맥락 설명을 생성합니다.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화 (컨텍스트 생성용 - 가벼운 모델 사용)
context_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Anthropic의 컨텍스트 생성 프롬프트 (한국어 버전)
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서의 청크에 맥락을 추가하는 전문가입니다.
주어진 청크가 전체 문서에서 어떤 위치에 있고 무엇에 대한 내용인지 간결하게 설명하세요.
설명은 50-100자 이내로 작성하세요.
오직 맥락 설명만 출력하세요."""),
    ("user", """<document>
{whole_document}
</document>

위 문서에서 아래 청크의 맥락을 설명해주세요:

<chunk>
{chunk_content}
</chunk>

맥락 설명:""")
])

# 컨텍스트 생성 체인
context_chain = context_prompt | context_llm | StrOutputParser()

print("컨텍스트 생성 체인 준비 완료")

컨텍스트 생성 체인 준비 완료


In [8]:
from langchain_core.documents import Document

# 각 청크에 대해 컨텍스트 생성
def generate_contexts(chunks: List[Document], whole_document: str) -> List[str]:
    """청크들에 대한 컨텍스트를 배치로 생성합니다."""
    inputs = [
        {"whole_document": whole_document, "chunk_content": chunk.page_content}
        for chunk in chunks
    ]
    
    # 배치 처리로 효율적으로 생성
    contexts = context_chain.batch(
        inputs, 
        #config={"max_concurrency": 5, "callbacks": [langfuse_handler]}
    )
    
    return contexts

# 컨텍스트 생성 실행
print("컨텍스트 생성 중...")
contexts = generate_contexts(chunks, body_docs)

print(f"\n생성된 컨텍스트 수: {len(contexts)}")
print("="*80)

for i, (chunk, context) in enumerate(zip(chunks[:5], contexts[:5])):
    print(f"\n[청크 {i+1}]")
    print(f"원본: {chunk.page_content[:100]}...")
    print(f"컨텍스트: {context}")

컨텍스트 생성 중...


OpenAIContextOverflowError: Error code: 400 - {'error': {'message': "This model's maximum context length is 1047576 tokens. However, your messages resulted in 1498944 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [14]:
# 컨텍스트가 추가된 청크 생성
contextual_chunks = []

for i, (chunk, context) in enumerate(zip(chunks, contexts)):
    # 컨텍스트 + 원본 청크 결합
    contextual_content = f"[맥락] {context}\n\n{chunk.page_content}"
    
    contextual_chunk = Document(
        page_content=contextual_content,
        metadata={
            **chunk.metadata,
            "chunk_id": i,
            "original_content": chunk.page_content,
            "context": context,
        }
    )
    contextual_chunks.append(contextual_chunk)

print(f"Contextual 청크 생성 완료: {len(contextual_chunks)}개")
print("="*80)

# 예시 출력
print("\n[Contextual 청크 예시]")
print(contextual_chunks[3].page_content)

Contextual 청크 생성 완료: 231개

[Contextual 청크 예시]
[맥락] 2026년 하반기 전기전자 산업 전망 중 AI 부품 병목 현상과 삼성전기의 구조적 경쟁력 및 투자 매력도를 분석한 부분입니다.

02
이번엔구조적
• 이중병목노출: FC-BGA(사실상완판)·MLCC(고부가믹스)를단일기업으로동시보유
• 이익가시성: LTA 기반가동률·판가락인으로이익변동성축소, 실적예측력제고
• 밸류에이션: 가시성제고는29년멀티플리레이팅의근거, 목표주가상향정당화
03
Top Pick
: 삼성전기


`(3) 하이브리드 검색 및 평가`

Embedding + BM25 + Reranker 파이프라인을 구성하고 성능을 평가합니다.

In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 일반 청크 벡터 저장소 (비교용)
normal_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="normal_chunks",
    persist_directory="./chroma_db"
)

# Contextual 청크 벡터 저장소
contextual_vectorstore = Chroma.from_documents(
    documents=contextual_chunks,
    embedding=embeddings,
    collection_name="contextual_chunks",
    persist_directory="./chroma_db"
)

print("벡터 저장소 생성 완료")
print(f"- 일반 청크: {normal_vectorstore._collection.count()}개")
print(f"- Contextual 청크: {contextual_vectorstore._collection.count()}개")

벡터 저장소 생성 완료
- 일반 청크: 237개
- Contextual 청크: 237개


In [16]:
# 테스트 쿼리
test_queries = [
    "삼성전자의 성장세는 어떠한가?",
    "바이오 업계 현황은?",
    "건설 업 현황은?",
    "MLCC란?",
]

# Retriever 생성
normal_retriever = normal_vectorstore.as_retriever(search_kwargs={"k": 3})
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 3})

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 검색
    normal_results = normal_retriever.invoke(query)
    print(f"\n[일반 검색 결과]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual 검색
    contextual_results = contextual_retriever.invoke(query)
    print(f"\n[Contextual 검색 결과]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '삼성전자의 성장세는 어떠한가?'

[일반 검색 결과]
  1. Mirae Asset Securities Research
11 | 2026 하반기전망_ 전기전자
폭발하는수요와제한적인공급에서발생하는프리미엄
병목에서발생하는프리미엄: FC-BGA &...
  2. 75.3 
36.3 
20.8 
10.3 
8.8 
7.4 
5.6 
8.4 
26.3 
19.7 
13.3 
평균
37.4 
176.9 
757.0 
180.5 
67.3 
38...
  3. Mirae Asset Securities Research
37 | 2026 하반기전망_ 전기전자
삼성전기(009150)
예상 포괄손익계산서 (요약)
예상 재무상태표 (요약)
예상 ...

[Contextual 검색 결과]
  1. [맥락] 2026년 하반기 전기전자 산업 내 삼성전기 매출 실적 및 부문별(광학, 전장, 기판) 분기별 전망 데이터 표입니다.

Mirae Asset Securities Resea...
  2. [맥락] 2026년 전기전자 산업 내 FC-BGA와 MLCC 부품 수급 병목 현상과 관련 기업들의 밸류에이션 현황을 분석하는 중간 부분입니다.

Mirae Asset Securit...
  3. [맥락] 삼성전기 실적 및 영업이익, 순이익 추이와 목표주가 산정 관련 재무지표와 성장률 분석 내용입니다.

7.1
8.7
11.6
13.2
12.5
7.8
11.6
20.7
컴포...

쿼리: '바이오 업계 현황은?'

[일반 검색 결과]
  1. MiraeAsset Securities Research96| 2026 하반기전망_ 제약/바이오시밀러에대한우호적환경조성중셀트리온(207940)투
•자사주소각에따른주식수감소를반영해목표...
  2. • 북미고객사프리미엄모델판매호조지속, 차기모델가변조리개탑재카메라모듈출하기대
투자의견(유지) 
목표주가(상향)
현재주가(26/6/10)
상승여력
1,275
Consensus 영업이익...
  3. MiraeAsset Securities Research50| 20

## BM25 Retriever 설정

In [17]:
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

# Kiwi 한국어 형태소 분석기 초기화
kiwi = Kiwi()

# 사용자 정의 단어 추가 (고유명사)
kiwi.add_user_word('반도체', 'NNP')
kiwi.add_user_word('바이오', 'NNP')
kiwi.add_user_word('건설', 'NNP')

# 한국어 토크나이저 전처리 함수
def kiwi_preprocess_func(text):
    """Kiwi 형태소 분석기를 사용한 토큰화"""
    return [t.form for t in kiwi.tokenize(text)]

# 일반 BM25 Retriever (Kiwi 토크나이저 적용)
normal_bm25 = BM25Retriever.from_documents(
    documents=chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

# Contextual BM25 Retriever (Kiwi 토크나이저 적용)
contextual_bm25 = BM25Retriever.from_documents(
    documents=contextual_chunks,
    preprocess_func=kiwi_preprocess_func,
    k=3
)

print("BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)")

BM25 Retriever 생성 완료 (Kiwi 토크나이저 적용)


In [18]:
# 정확한 키워드가 필요한 쿼리
keyword_queries = [
    "리가켐바이오",
    "바이오시밀러",
    "MLCC",
    "메모리",
]

for query in keyword_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 BM25
    normal_results = normal_bm25.invoke(query)
    print(f"\n[일반 BM25]")
    for i, doc in enumerate(normal_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    # Contextual BM25
    contextual_results = contextual_bm25.invoke(query)
    print(f"\n[Contextual BM25]")
    for i, doc in enumerate(contextual_results):
        print(f"  {i+1}. {doc.page_content[:100]}...")


쿼리: '리가켐바이오'

[일반 BM25]
  1. MiraeAsset Securities Research108| 2026 하반기전망_ 제약/바이오Compliance
0
20,000
40,000
60,000
80,000
24.6
2...
  2. MiraeAsset Securities Research26| 2026 하반기전망_ 제약/바이오I. R&D -항암
•리가켐바이오, 현재까지J&J, Amgen, Takeda 등총15건...
  3. Q. 2027년DRAM과NAND 공급부족은올해보다더심해지는가?A. 2027년전반에도수급이타이트할것이며, 이는20...

[Contextual BM25]
  1. [맥락] 2026년 제약/바이오 산업 전망 보고서 내 리가켐바이오의 항암 분야 연구개발 및 기술이전 성과와 파트너십 현황을 다룬 부분입니다.

MiraeAsset Securitie...
  2. [맥락] 2026년 하반기 제약/바이오 섹터 내 주요 기업들의 주가 추이 및 컴플라이언스 관련 내용을 다룬 부분입니다.

MiraeAsset Securities Research10...
  3. [맥락] 2026년 하반기 제약/바이오 섹터 내 리가켐바이오의 기술이전 성과와 파트너십, 기술수출 전망을 다룬 종목 분석 부분입니다.

MiraeAsset Securities Re...

쿼리: '바이오시밀러'

[일반 BM25]
  1. MiraeAsset Securities Research50| 2026 하반기전망_ 제약/바이오
•바이오시밀러임상3상비교효능시험일반적으로1~3년, $24mn 소요-> 임상시험간소화바...
  2. MiraeAsset Securities Research96| 2026 하반기전망_ 제약/바이오시밀러에대한우호적환경조성중셀트리온(207940)투
•자사주소각에따른주식수감소를반영해목표...
  3. MiraeAsset Securities Research51| 2026 하반기전망_ 제약/바이오주요국가시밀러관련우호정책지역정책/ 제도시점·진행상태핵심내용바이오시밀러영향

## 하이브리드 검색 (Embedding + BM25)

In [19]:
from langchain_classic.retrievers import EnsembleRetriever

# 일반 하이브리드 Retriever
normal_hybrid = EnsembleRetriever(
    retrievers=[normal_retriever, normal_bm25],
    weights=[0.5, 0.5],  # Embedding과 BM25 동일 가중치
)

# Contextual 하이브리드 Retriever
contextual_hybrid = EnsembleRetriever(
    retrievers=[contextual_retriever, contextual_bm25],
    weights=[0.5, 0.5],
)

print("하이브리드 Retriever 생성 완료")

하이브리드 Retriever 생성 완료


In [20]:
# 다양한 유형의 쿼리
hybrid_queries = [
    "AI 산업에 대한 전망과 수요는?",          
    "전쟁으로 인해 건설 주가 오를것 같나요?", 
    "바이오 주식의 전망?",     # 혼합
]

for query in hybrid_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # 일반 하이브리드
    normal_results = normal_hybrid.invoke(query)
    print(f"\n[일반 하이브리드] 결과 {len(normal_results)}개")
    for i, doc in enumerate(normal_results[:2]):
        print(f"  {i+1}. {doc.page_content[:80]}...")
    
    # Contextual 하이브리드
    contextual_results = contextual_hybrid.invoke(query)
    print(f"\n[Contextual 하이브리드] 결과 {len(contextual_results)}개")
    for i, doc in enumerate(contextual_results[:2]):
        # 원본 내용 표시
        original = doc.metadata.get('original_content', doc.page_content[:80])
        print(f"  {i+1}. {original[:80]}...")


쿼리: 'AI 산업에 대한 전망과 수요는?'

[일반 하이브리드] 결과 5개
  1. Mirae Asset Securities Research
30 | 2026 하반기전망_ 전기전자
0
2,000
4,000
6,000
8,000
...
  2. Mirae Asset Securities Research
10 | 2026 하반기전망_ 전기전자
AI데이터센터밸류체인
AI 데이터센터밸류체인: ...

[Contextual 하이브리드] 결과 6개
  1. MiraeAsset Securities Research1| 2026 하반기전망_ 제약/바이오모두가바이오반등을위해싸우고있다제약...
  2. Q. 2027년DRAM과NAND 공급부족은올해보다더심해지는가?A. 2027년전반에도수급이타이트할것이며, 이는20...

쿼리: '전쟁으로 인해 건설 주가 오를것 같나요?'

[일반 하이브리드] 결과 6개
  1. MiraeAsset Securities Research32| 2026 하반기전망_ 건설/건자재
0
50,000
100,000
150,000
20...
  2. Q. 2027년DRAM과NAND 공급부족은올해보다더심해지는가?A. 2027년전반에도수급이타이트할것이며, 이는20...

[Contextual 하이브리드] 결과 6개
  1. MiraeAsset Securities Research32| 2026 하반기전망_ 건설/건자재
0
50,000
100,000
150,000
20...
  2. Q. 2027년DRAM과NAND 공급부족은올해보다더심해지는가?A. 2027년전반에도수급이타이트할것이며, 이는20...

쿼리: '바이오 주식의 전망?'

[일반 하이브리드] 결과 6개
  1. MiraeAsset Securities Research2| 2026 하반기전망_ 제약/바이오
CONTENTS[요약] 모두가바이오반등을위해싸우고있...
  2. 마이크론주요Q&AQ. SCA에서5년간하한가격으로보장되는매출규모는어느정도인가?A. 현재까지체결된SCA(...

[Contextual 하이브리드

## Reranker 추가

하이브리드 검색 결과를 **Reranker로 재순위화**하여 최종 성능을 극대화합니다.

In [21]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Cross-Encoder 모델 초기화
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

# Contextual Hybrid + Reranker
contextual_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=contextual_hybrid,
)

print("Reranker 설정 완료")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6546.29it/s]


Reranker 설정 완료


In [22]:
# 최종 검색 테스트
final_queries = [
    "AI 산업에 대한 전망과 수요는?",          
    "전쟁으로 인해 건설 주가 오를것 같나요?", 
    "바이오 주식의 전망?",     # 혼합
]


for query in final_queries:
    print(f"\n{'='*80}")
    print(f"쿼리: '{query}'")
    print("="*80)
    
    # Contextual Hybrid + Reranker
    results = contextual_rerank_retriever.invoke(query, config={"callbacks": [langfuse_handler]})
    
    print(f"\n[Contextual Hybrid + Reranker] Top {len(results)}개")
    for i, doc in enumerate(results):
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', 'N/A')
        print(f"\n  [{i+1}] 맥락: {context}")
        print(f"      내용: {original[:100]}...")


쿼리: 'AI 산업에 대한 전망과 수요는?'

[Contextual Hybrid + Reranker] Top 3개

  [1] 맥락: 2026년 전기전자 산업 내 AI 서버용 FC-BGA 기판 증설과 솔더볼 수요 증가 전망을 다룬 중간부문 분석 내용입니다.
      내용: Mirae Asset Securities Research
17 | 2026 하반기전망_ 전기전자
1
1.4
1.7
2
2.3
2.8
0
0.5
1
1.5
2
2.5
3
2023
2...

  [2] 맥락: 문서 초반부 요약 섹션으로, 2026년 하반기 전기전자 산업 내 AI 부품 병목 현상과 자본 이동, 공급 구조 변화를 분석한 내용입니다.
      내용: Mirae Asset Securities Research
3 | 2026 하반기전망_ 전기전자
[요약] AI 부품도병목이다
병목에돈이흐른다
• 수요동인: 학습→ 추론전환으로토큰·전...

  [3] 맥락: 2026년 하반기 제약·바이오 산업 전망 보고서의 서두 요약 부분으로, 바이오 산업 반등을 위한 현황과 전망을 소개합니다.
      내용: MiraeAsset Securities Research4| 2026 하반기전망_ 제약/바이오[요약] 모두가바이오반등을위해싸우...

쿼리: '전쟁으로 인해 건설 주가 오를것 같나요?'

[Contextual Hybrid + Reranker] Top 3개

  [1] 맥락: 2026년 하반기 건설/건자재 섹터 내 주요 건설사별 주가 추이 및 실적 전망 그래프와 데이터 분석 부분입니다.
      내용: MiraeAsset Securities Research32| 2026 하반기전망_ 건설/건자재
0
50,000
100,000
150,000
200,000
250,000
24.6
2...

  [2] 맥락: 2026년 하반기 건설/건자재 섹터 내 현대건설의 해외 원전 수주 및 사업 성과와 투자 현황을 다룬 종목 분석 부분입니다.
      내용: MiraeAsset Securities Researc

In [23]:
from langchain_core.runnables import RunnablePassthrough

# 답변 생성용 LLM
answer_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 질의응답 전문가입니다.
주어진 문맥을 바탕으로 질문에 정확하게 답변하세요.
문맥에 없는 내용은 "문서에서 해당 정보를 찾을 수 없습니다."라고 답변하세요."""),
    ("user", """문맥:
{context}

질문: {question}

답변:""")
])

# 문서 포맷팅 함수
def format_docs(docs):
    formatted = []
    for doc in docs:
        # 원본 내용과 맥락 모두 포함
        original = doc.metadata.get('original_content', doc.page_content)
        context = doc.metadata.get('context', '')
        if context:
            formatted.append(f"[맥락: {context}]\n{original}")
        else:
            formatted.append(original)
    return "\n\n---\n\n".join(formatted)

# RAG 체인 구성
rag_chain = (
    {"context": contextual_rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | answer_llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")

RAG 체인 구성 완료


In [24]:
# RAG 체인 실행
questions = [
    "전자분야 주식 전망에 대해 알려주세요",
    "바이오 분야 현황이 어떤가요?",
    "건설주는 어떤가요?",
]

for question in questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print("-"*80)
    
    answer = rag_chain.invoke(question, config={"callbacks": [langfuse_handler]})
    print(f"A: {answer}")


Q: 전자분야 주식 전망에 대해 알려주세요
--------------------------------------------------------------------------------
A: 문서에서 전자분야 주식 전망에 대한 구체적인 내용은 제공되지 않았습니다.

Q: 바이오 분야 현황이 어떤가요?
--------------------------------------------------------------------------------
A: 문서에서 바이오 분야 현황에 대해 구체적으로 설명한 부분은 제한적이나, 2026년 하반기 제약·바이오 산업 전망 보고서 요약에서는 "모두가 바이오 반등을 위해 싸우고 있다"는 표현으로 바이오 산업의 반등을 위한 노력이 활발함을 시사하고 있습니다. 또한, 바이오시밀러 임상시험이 1~3년, 약 2,400만 달러가 소요되나 임상시험 간소화가 진행 중이며, FDA 허가를 equivalence(동등성)의 기준선으로 인식하는 등 정책적·시장 경쟁 환경이 변화하고 있음을 알 수 있습니다. 전반적으로 바이오 산업은 경쟁 심화 속에서도 오리지널 제품 매출 흡수 가능성과 제조 원가 경쟁력 확보를 통해 반등을 모색하는 상황으로 보입니다.

Q: 건설주는 어떤가요?
--------------------------------------------------------------------------------
A: 2026년 하반기 건설주에 대한 구체적인 주가 추이와 실적 전망이 현대건설, GS건설, 삼성E&A, 대우건설, DL이앤씨 등 주요 건설사별로 제시되어 있습니다. 다만, 이란 및 중동 사태로 인해 국내 자재 수급에 단기적인 불안 요인이 존재하며, 해외 중동 공사 현장에서는 자재 수급과 인력 투입 문제로 공사 지연 가능성도 있습니다. 따라서 건설주는 전반적으로 자재 수급과 공사 지연 리스크를 안고 있으나, 주요 건설사별 밸류에이션 비교와 주가 추이 분석을 통해 투자 판단이 필요합니다. 

요약하면, 2026년 하반기 건설주는 자

In [ ]:
# 다양한 Retriever 성능 비교 (Kiwi 기반)
from kiwipiepy import Kiwi

retrievers_to_compare = {
    "일반 Embedding": normal_retriever,
    "일반 BM25": normal_bm25,
    "일반 Hybrid": normal_hybrid,
    "Contextual Embedding": contextual_retriever,
    "Contextual BM25": contextual_bm25,
    "Contextual Hybrid": contextual_hybrid,
    "Contextual + Reranker": contextual_rerank_retriever,
}

# 테스트 쿼리와 기대 키워드
# 핵심: 컨텍스트에만 있고 원본 청크에는 없는 키워드로 검색
# k=1로 설정하여 정확한 청크 매칭 테스트
test_queries = [
    # 직접적인 쿼리 (일반 검색도 가능) - 원본 청크에 키워드 존재
    {"query": "에이비엘의 IGF1R 타깃 강점은?", "expected": "Grabody-B 높은 뇌 발현도 IGF1R 32.7%"},
    {"query": "비만 신약 siRNA란?", "expected": "1,000만 셀"},
    
    # 맥락 의존적 쿼리 - 컨텍스트에만 있는 표현 사용
    # 원본 청크: "FSD 베타 버전이... Dojo 슈퍼컴퓨터를 활용한 AI 학습"
    # 컨텍스트: "기술 개발 부문, 자율주행과 AI 학습 관련 최신 성과"
    {"query": "테슬라의 최신 성과와 진행 상황", "expected": "Dojo"},
    
    # 원본 청크: "4680 배터리 셀의 대량 생산... kWh당 100달러"
    # 컨텍스트: "기술 혁신과 미래 전략의 핵심 내용"
    {"query": "테슬라의 기술 혁신 핵심 내용", "expected": "100달러"},
    
    # 원본 청크: "2023년 테슬라의 총 매출은 967억 달러... 에너지 저장 부문"
    # 컨텍스트: "재무 성과와 매출, 생산량에 대한 핵심 내용을 요약"
    {"query": "테슬라 재무 성과 요약", "expected": "967억 달러"},
    
    # 원본 청크: "Tesla, Inc.는 2003년 설립... 일론 머스크가 CEO"
    # 컨텍스트: "회사의 설립 배경, 주요 사업 분야"
    {"query": "테슬라 설립 배경과 사업 분야", "expected": "2003년"},
]

kiwi = Kiwi()

def evaluate_contextual_retrievers(retrievers: dict, test_queries: list, k: int = 1):
    """Contextual vs Non-Contextual Retriever 성능 비교
    
    k=1로 설정하여 가장 관련성 높은 1개 청크만 검색.
    문서가 작을 때 (6개 청크) k=3이면 50%를 검색하므로 변별력이 낮음.
    """
    results_dict = {}
    
    for name, retriever in retrievers.items():
        correct = 0
        total = len(test_queries)
        
        for test in test_queries:
            query = test["query"]
            expected = test["expected"]
            
            try:
                results = retriever.invoke(query)[:k]
                all_content = " ".join([
                    doc.metadata.get('original_content', '') or doc.page_content 
                    for doc in results
                ])
                
                # 기대 키워드가 검색 결과에 포함되어 있는지 확인
                if expected in all_content:
                    correct += 1
            except Exception as e:
                pass
        
        accuracy = correct / total
        results_dict[name] = {"accuracy": accuracy, "correct": correct, "total": total}
    
    return results_dict

print("="*80)
print("Contextual Retrieval 성능 비교 (k=1, Top-1 정확도)")
print("="*80)

eval_results = evaluate_contextual_retrievers(retrievers_to_compare, test_queries, k=1)

for name, result in eval_results.items():
    status = "✅" if result["accuracy"] >= 0.5 else "❌"
    print(f"{status} {name:25s}: {result['accuracy']:.1%} ({result['correct']}/{result['total']})")

# 일반 vs Contextual 비교
print("\n" + "="*80)
print("일반 vs Contextual 비교 요약")
print("="*80)

normal_avg = sum([eval_results[k]["accuracy"] for k in ["일반 Embedding", "일반 BM25", "일반 Hybrid"]]) / 3
contextual_avg = sum([eval_results[k]["accuracy"] for k in ["Contextual Embedding", "Contextual BM25", "Contextual Hybrid"]]) / 3
best_result = eval_results["Contextual + Reranker"]["accuracy"]

print(f"일반 검색 평균 정확도:      {normal_avg:.1%}")
print(f"Contextual 검색 평균 정확도: {contextual_avg:.1%}")
print(f"Contextual + Reranker:      {best_result:.1%}")

if normal_avg > 0:
    improvement = ((contextual_avg - normal_avg) / normal_avg * 100)
    print(f"\n→ Contextual 방식이 일반 방식 대비 {improvement:.1f}% 향상")
else:
    print(f"\n→ 일반 검색 정확도가 0%이므로 비율 계산 불가")